# Dida365/TickTick API Client Demo

This notebook demonstrates how to use the Dida365/TickTick API client to interact with your tasks and projects.

## Setup

First, we need to set up our environment and authenticate with the API.

In [5]:
import asyncio
from dida365 import (
    Dida365Client,
    ServiceType,
    ProjectCreate,
    ProjectUpdate,
    TaskCreate,
    TaskUpdate,
    TaskPriority,
    TaskStatus,
    ViewMode
)

# Get you TickTick/DIDA365 credentials from:
# https://developer.ticktick.com/manage
# https://developer.dida365.com/manage
# Replace these with your OAuth2 credentials
# Check README.md  "OAuth2 Setup"
CLIENT_ID = ""
CLIENT_SECRET = ""


# Create client instance 
client = Dida365Client(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    service_type=ServiceType.TICKTICK, # or ServiceType.DIDA365
    save_to_env=False,
)

# client = Dida365Client()

## Authentication

The client uses OAuth2 for authentication. When you run the cell below, it will open your browser for authorization.

In [ ]:
async def authenticate():
    # This will open your browser for authorization
    token_info = await client.authenticate()
    print("Successfully authenticated!")
    return token_info

await authenticate()

## Projects

Let's start by working with projects. We'll demonstrate how to:
1. Create a new project
2. List all projects
3. Update a project
4. Get project details
5. Delete a project

In [ ]:
# Create a new project
new_project = ProjectCreate(
    name="Demo Project",
    color="#FF0000",  # Red color
    view_mode=ViewMode.KANBAN
)
project = await client.create_project(new_project)
print(f"Created project: {project.name} (ID: {project.id})")


In [ ]:

# List all projects
projects = await client.get_projects()
print("\nAll projects:")
for p in projects:
    print(f"- {p.name} (ID: {p.id})")
    


In [ ]:
# Get active projects (filtering out closed projects)
active_projects = [p for p in projects if not p.closed]
print("\nActive projects:")
for p in active_projects:
    print(f"- {p.name} (ID: {p.id})")



In [ ]:

# Update the project
update = ProjectUpdate(
    id=project.id,
    name="Updated Demo Project",
    color="#00FF00"  # Green color
)
updated = await client.update_project(update)
print(f"\nUpdated project name to: {updated.name}")


In [ ]:
# Get project details with tasks
details = await client.get_project_with_data(project.id)
print(f"\nProject details:")
print(f"Name: {details.project.name}")
print(f"Tasks: {len(details.tasks)}")

# Delete the project
await client.delete_project(project.id)
print("\nProject deleted successfully")

## Tasks

Now let's work with tasks. We'll demonstrate how to:
1. Create a new task
2. Get task details
3. Update a task
4. Complete a task
5. Delete a task

In [ ]:
# First create a project for our tasks
project = await client.create_project(ProjectCreate(name="Task Demo Project"))

# Create a new task
new_task = TaskCreate(
    title="Important Demo Task",
    content="This is a demo task with high priority",
    project_id=project.id,
    priority=TaskPriority.HIGH
)
task = await client.create_task(new_task)
print(f"Created task: {task.title} (ID: {task.id})")


In [ ]:
# Get task details
task_details = await client.get_task(project.id, task.id)
print(f"\nTask details:")
print(f"Title: {task_details.title}")
print(f"Priority: {task_details.priority}")
print(f"Status: {task_details.status}")


In [ ]:
# Update the task
update = TaskUpdate(
    id=task.id,
    project_id=project.id,
    title="Updated Demo Task",
    priority=TaskPriority.MEDIUM
)
updated = await client.update_task(update)
print(f"\nUpdated task title to: {updated.title}")


In [ ]:

# Complete the task
await client.complete_task(project.id, task.id)
print("\nTask marked as completed")


In [ ]:

# Delete the task
await client.delete_task(project.id, task.id)
print("\nTask deleted successfully")


In [19]:

# Clean up by deleting the project
await client.delete_project(project.id)

## Error Handling

The API client includes comprehensive error handling. Let's demonstrate how different types of errors are handled.

In [ ]:
from dida365.exceptions import NotFoundError, ValidationError

async def demo_error_handling():
    try:
        # Try to get a non-existent project
        await client.get_project("non_existent_id")
    except NotFoundError as e:
        print(f"Not Found Error: {e}")
    
    try:
        # Try to create a project with invalid data
        await client.create_project(ProjectCreate(
            name="Invalid Project",
            color="invalid_color"  # Invalid color format
        ))
    except ValidationError as e:
        print(f"\nValidation Error: {e}")

await demo_error_handling()

## Cleanup

Always remember to close the client when you're done to clean up resources.

In [ ]:
await client.http.close()
print("Client closed successfully")